# Data acquisition and cleaning

Acquire immutable daily snapshots, validate their source representations, and export SPY-aligned prices, cash returns, VIX and risk-free holding intervals.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO
import csv
import hashlib
import importlib.metadata as metadata
import json
import platform
import tempfile
import time

import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.api.types import is_datetime64_any_dtype
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "requirements.txt").is_file()
             and (p / "notebooks").is_dir() and (p / "scripts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project directory or a subdirectory.")
NOTEBOOK_DIR = ROOT / "notebooks"
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data/interim"
SOURCE_REPORTS = NOTEBOOK_DIR / "quality_checks"
CLEANING_REPORTS = NOTEBOOK_DIR / "cleaning_checks"
for folder in [RAW, INTERIM, ROOT / "data/processed", SOURCE_REPORTS, CLEANING_REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

START = "2006-01-01"
END_INCLUSIVE = "2026-09-30"
END_EXCLUSIVE = "2026-10-01"
START_DATE, END_DATE = pd.Timestamp(START), pd.Timestamp(END_INCLUSIVE)
NY = "America/New_York"
XLF_EVENT = pd.Timestamp("2016-09-19")
SUFFIX = f"{START}_{END_INCLUSIVE}"
RULE_VERSION = "02-cleaning-v1.1"
ETFS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY",
               "SPY", "TLT", "IEF", "GLD", "HYG", "LQD"]
SECTORS = ETFS[:9]
RANDOM_SEED = 20261020
DOWNLOAD_MANIFEST_PATH = NOTEBOOK_DIR / "download_manifest.json"
VERSIONS = {p: metadata.version(p) for p in
            ["yfinance", "pandas", "numpy", "requests", "curl_cffi", "beautifulsoup4", "nbformat", "nbclient"]}
pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:.8g}")
print(f"Daily observations: {START} to {END_INCLUSIVE}; Python {platform.python_version()}")

YF_CACHE = tempfile.TemporaryDirectory(prefix="dsa5205_yfinance_")
yf.set_tz_cache_location(YF_CACHE.name)
yf.config.debug.hide_exceptions = False
yf.config.network.retries = 2
YAHOO_SESSION = curl_requests.Session(impersonate="chrome")
HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update({"User-Agent": "DSA5205-academic-research/1.0"})

if DOWNLOAD_MANIFEST_PATH.exists():
    download_manifest = json.loads(DOWNLOAD_MANIFEST_PATH.read_text(encoding="utf-8"))
    if download_manifest["requested_start"] != START or download_manifest["requested_end_inclusive"] != END_INCLUSIVE:
        raise RuntimeError("The existing manifest covers a different date range.")
else:
    download_manifest = {"requested_start": START, "requested_end_inclusive": END_INCLUSIVE,
                "created_utc": datetime.now(timezone.utc).isoformat(),
                "python": platform.python_version(), "versions": VERSIONS,
                "files": {}, "failures": [], "notes": []}
download_failures = []


## Data acquisition

Existing snapshots are verified against the download manifest before reuse.

In [ ]:
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256(path):
    return sha256_bytes(path.read_bytes())

def persist_manifest():
    download_manifest["last_run_utc"] = datetime.now(timezone.utc).isoformat()
    download_manifest["failures"] = download_failures
    DOWNLOAD_MANIFEST_PATH.write_text(json.dumps(download_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

def save_once(path, payload, *, asset, kind, source_url, extra=None):
    digest = sha256_bytes(payload)
    if path.exists():
        if sha256_bytes(path.read_bytes()) != digest:
            raise RuntimeError(f"Refusing to overwrite a different raw snapshot: {path.name}")
    else:
        with path.open("xb") as f:
            f.write(payload)
    download_manifest["files"][path.name] = {
        "asset": asset, "kind": kind, "source_url": source_url,
        "saved_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": digest, "bytes": len(payload), **(extra or {})}
    persist_manifest()

def verify_existing(path):
    if not path.exists():
        return False
    entry = download_manifest["files"].get(path.name)
    if entry is None:
        raise RuntimeError(f"Raw snapshot has no manifest entry: {path.name}")
    if sha256_bytes(path.read_bytes()) != entry["sha256"]:
        raise RuntimeError(f"Raw snapshot checksum mismatch: {path.name}")
    return True

def get_response(url, *, params=None, yahoo=False, attempts=4):
    session = YAHOO_SESSION if yahoo else HTTP_SESSION
    for attempt in range(attempts):
        response = None
        try:
            response = session.get(url, params=params, timeout=35)
            response.raise_for_status()
            return response
        except Exception:
            status = getattr(response, "status_code", None)
            retryable = status is None or status == 429 or status >= 500
            if not retryable or attempt == attempts - 1:
                raise
            delay = min(2 ** (attempt + 1), 12)
            print(f"Request failed (HTTP {status or 'network error'}); retrying in {delay}s: {url}", flush=True)
            time.sleep(delay)

def record_failure(asset, stage, error):
    item = {"asset": asset, "stage": stage, "error": f"{type(error).__name__}: {error}"}
    download_failures.append(item)
    persist_manifest()
    print("Download failed: ", item, flush=True)


### ETF prices and corporate actions

In [ ]:
PERIOD1 = int(pd.Timestamp(START, tz=NY).timestamp())
PERIOD2 = int(pd.Timestamp(END_EXCLUSIVE, tz=NY).timestamp())

def etf_path(ticker, suffix):
    return RAW / f"{ticker}_{START}_{END_INCLUSIVE}_{suffix}"

def download_etf(ticker):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    params = {"period1": PERIOD1, "period2": PERIOD2, "interval": "1d",
              "events": "div,splits,capitalGains", "includePrePost": "false"}
    json_path = etf_path(ticker, "yahoo_chart.json")
    if not verify_existing(json_path):
        response = get_response(url, params=params, yahoo=True)
        payload = response.json()
        chart = payload.get("chart", {})
        if chart.get("error") or not chart.get("result"):
            raise ValueError(f"Yahoo chart error: {chart.get('error')}")
        result = chart["result"][0]
        if not result.get("timestamp"):
            raise ValueError("Yahoo returned an empty time series.")
        save_once(json_path, response.content, asset=ticker, kind="yahoo_chart_json",
                  source_url=str(response.url), extra={"request_params": params})
    ticker_obj = yf.Ticker(ticker, session=YAHOO_SESSION)
    for adjusted in [False, True]:
        suffix = "yfinance_adjusted.csv" if adjusted else "yfinance_raw.csv"
        path = etf_path(ticker, suffix)
        if verify_existing(path):
            continue
        frame = None
        for attempt in range(4):
            try:
                frame = ticker_obj.history(
                    start=START, end=END_EXCLUSIVE, interval="1d", actions=True,
                    auto_adjust=adjusted, back_adjust=False, repair=False,
                    keepna=True, rounding=False, timeout=35)
                if frame.empty or not {"Open", "High", "Low", "Close", "Volume"}.issubset(frame.columns):
                    raise ValueError("yfinance returned no data or omitted required OHLCV fields.")
                break
            except Exception:
                if attempt == 3:
                    raise
                time.sleep(min(2 ** (attempt + 1), 12))
        blob = frame.to_csv(index=True, index_label="Date").encode("utf-8")
        save_once(path, blob, asset=ticker,
                  kind="yfinance_adjusted" if adjusted else "yfinance_raw",
                  source_url=url,
                  extra={"auto_adjust": adjusted, "actions": True, "repair": False,
                         "keepna": True, "rows_at_download": len(frame),
                         "columns": list(frame.columns),
                         "date_semantics": "exchange-local daily date; timezone retained in CSV"})
        print(f"  Saved {path.name}: {len(frame):,} rows", flush=True)


In [ ]:
for number, ticker in enumerate(ETFS, start=1):
    print(f"[{number:02d}/{len(ETFS)}] {ticker}", flush=True)
    try:
        download_etf(ticker)
    except Exception as exc:
        record_failure(ticker, "ETF download", exc)


### VIX

In [ ]:
VIX_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"
VIX_PATH = RAW / f"VIX_{START}_{END_INCLUSIVE}_cboe.csv"
try:
    if not verify_existing(VIX_PATH):
        response = get_response(VIX_URL)
        text = response.content.decode("utf-8-sig")
        lines = text.splitlines(keepends=True)
        if not lines or "DATE,OPEN,HIGH,LOW,CLOSE" not in lines[0].upper():
            raise ValueError("Unexpected Cboe VIX response.")
        selected = [lines[0]]
        for line in lines[1:]:
            if not line.strip():
                continue
            date_text = next(csv.reader([line]))[0]
            day = pd.to_datetime(date_text, format="%m/%d/%Y").strftime("%Y-%m-%d")
            if START <= day <= END_INCLUSIVE:
                selected.append(line)
        if len(selected) <= 1:
            raise ValueError("No VIX observations in the requested range.")
        save_once(VIX_PATH, "".join(selected).encode("utf-8"), asset="VIX", kind="cboe_csv",
                  source_url=str(response.url),
                  extra={"full_response_sha256": sha256_bytes(response.content),
                         "range_selection": "original CSV lines only; no value edits",
                         "source_bytes": len(response.content)})
    print("VIX snapshot: ", VIX_PATH.name, flush=True)
except Exception as exc:
    record_failure("VIX", "Cboe download", exc)


### Treasury yield

In [ ]:
FRED_CSV_URL = "https://fred.stlouisfed.org/graph/graph.csv"
FRED_TABLE_URL = "https://fred.stlouisfed.org/data/DGS3MO"
FRED_PATH = RAW / f"DGS3MO_{START}_{END_INCLUSIVE}_fred.csv"
try:
    if not verify_existing(FRED_PATH):
        csv_error = None
        try:
            response = get_response(FRED_CSV_URL, params={"id": "DGS3MO", "cosd": START, "coed": END_INCLUSIVE}, attempts=1)
            text = response.content.decode("utf-8-sig")
            lines = text.splitlines(keepends=True)
            if not lines or "DGS3MO" not in lines[0] or "<html" in text[:200].lower():
                raise ValueError("The FRED CSV endpoint did not return DGS3MO data.")
            selected = [lines[0]]
            for line in lines[1:]:
                if not line.strip():
                    continue
                day = next(csv.reader([line]))[0]
                if START <= day <= END_INCLUSIVE:
                    selected.append(line)
            if len(selected) <= 1:
                raise ValueError("No FRED observations in the requested range.")
            blob = "".join(selected).encode("utf-8")
            method = "FRED CSV; original lines in requested range"
        except Exception as exc:
            csv_error = f"{type(exc).__name__}: {exc}"
            print("FRED CSV unavailable; using the official observation table.", flush=True)
            response = get_response(FRED_TABLE_URL)
            soup = BeautifulSoup(response.content, "html.parser")
            table = soup.find("table", id="data-table-observations")
            if table is None or "DGS3MO" not in soup.get_text():
                raise ValueError("The official FRED page has no DGS3MO observation table.")
            observations = []
            for tr in table.find_all("tr"):
                date_cell, value_cell = tr.find("th", scope="row"), tr.find("td")
                if date_cell is not None and value_cell is not None:
                    observations.append((date_cell.get_text().strip(), value_cell.get_text()))
            extra = soup.find(id="extra-rows")
            if extra is not None:
                for part in extra.get_text().split("#"):
                    if "|" in part:
                        day, value = part.split("|", 1)
                        observations.append((day.strip(), value))
            selected = [(day, value) for day, value in observations if START <= day <= END_INCLUSIVE]
            if not selected or max(day for day, _ in selected) < END_INCLUSIVE:
                raise ValueError("The FRED observation table does not cover the requested range.")
            stream = StringIO(newline="")
            writer = csv.writer(stream)
            writer.writerow(["observation_date", "DGS3MO"])
            writer.writerows(selected)
            blob = stream.getvalue().encode("utf-8")
            method = "FRED official HTML observation table + extra-rows; values/missing markers preserved"
        save_once(FRED_PATH, blob, asset="DGS3MO", kind="fred_csv", source_url=str(response.url),
                  extra={"acquisition_method": method, "csv_endpoint_error": csv_error,
                         "full_response_sha256": sha256_bytes(response.content),
                         "source_bytes": len(response.content), "units": "annualized percent",
                         "no_fill_or_rate_conversion": True})
    print("DGS3MO snapshot: ", FRED_PATH.name, flush=True)
except Exception as exc:
    record_failure("DGS3MO", "FRED download", exc)


### Acquisition coverage

In [ ]:
persist_manifest()
expected_paths = [etf_path(t, suffix) for t in ETFS for suffix in
                  ["yahoo_chart.json", "yfinance_raw.csv", "yfinance_adjusted.csv"]] + [VIX_PATH, FRED_PATH]
missing_files = [p.name for p in expected_paths if not p.exists()]
print(f"Raw snapshots: {len(expected_paths)-len(missing_files)}/{len(expected_paths)}")
if missing_files or download_failures:
    print("Missing files:", missing_files, "Download errors:", download_failures)
YAHOO_SESSION.close()
HTTP_SESSION.close()
YF_CACHE.cleanup()


## Source validation

Reconcile vendor representations and inspect prices, corporate actions and trading calendars.

In [ ]:
COMPARE_RTOL = 1e-10
COMPARE_ATOL = 1e-10

def compare_numeric_values(asset, field, dates, expected, actual, comparison):
    left = pd.to_numeric(expected, errors="coerce").reset_index(drop=True)
    right = pd.to_numeric(actual, errors="coerce").reset_index(drop=True)
    days = pd.Series(dates).reset_index(drop=True)
    finite = np.isfinite(left) & np.isfinite(right)
    null_changed = left.isna() ^ right.isna()
    # Volume is an integer count; price comparisons allow serialization noise.
    rtol, atol = (0.0, 0.0) if field == "Volume" else (COMPARE_RTOL, COMPARE_ATOL)
    close = pd.Series(np.isclose(left, right, rtol=rtol, atol=atol, equal_nan=True))
    bad = ~close
    delta = (left - right).abs()
    relative = delta / left.abs().where(left.ne(0))
    summary = {"asset": asset, "comparison": comparison, "field": field, "rows": len(days),
               "finite_pairs": int(finite.sum()), "expected_nulls": int(left.isna().sum()),
               "actual_nulls": int(right.isna().sum()), "null_mask_mismatches": int(null_changed.sum()),
               "different_rows": int(bad.sum()), "max_abs_difference": delta[finite].max(),
               "max_relative_difference": relative[finite].max(), "rtol": rtol, "atol": atol}
    details = pd.DataFrame({"asset": asset, "comparison": comparison, "field": field,
                            "date": days, "expected": left, "actual": right,
                            "absolute_difference": delta, "null_mask_changed": null_changed}).loc[bad]
    return summary, details

def date_pair(left, right, asset, comparison):
    left_dates, right_dates = set(left["Date"]), set(right["Date"])
    summary = {"asset": asset, "comparison": comparison,
               "left_rows": len(left), "right_rows": len(right),
               "left_duplicate_dates": int(left["Date"].duplicated().sum()),
               "right_duplicate_dates": int(right["Date"].duplicated().sum()),
               "left_only_dates": len(left_dates - right_dates), "right_only_dates": len(right_dates - left_dates)}
    if summary["left_duplicate_dates"] or summary["right_duplicate_dates"]:
        raise ValueError(f"{asset} {comparison} contains duplicate dates; one-to-one reconciliation is not possible.")
    joined = left.merge(right, on="Date", how="outer", suffixes=("_expected", "_actual"), validate="one_to_one")
    return joined, summary


### Adjusted OHLC

Check adjusted OHLC against the vendor factor `Adj Close / Close`.


In [ ]:
def check_adjusted_ohlc(saved):
    rows, detail_frames, date_rows = [], [], []
    for ticker in ETFS:
        raw_name = etf_path(ticker, "yfinance_raw.csv").name
        adj_name = etf_path(ticker, "yfinance_adjusted.csv").name
        if raw_name not in saved or adj_name not in saved:
            continue
        raw = saved[raw_name].copy()
        adjusted = saved[adj_name].copy()
        ratio = numeric(raw, "Adj Close") / numeric(raw, "Close")
        expected = pd.DataFrame({"Date": raw["Date"]})
        actual = pd.DataFrame({"Date": adjusted["Date"]})
        for field in ["Open", "High", "Low", "Close"]:
            expected[field] = numeric(raw, field) * ratio if field != "Close" else numeric(raw, "Adj Close")
            actual[field] = numeric(adjusted, field)
        joined, date_row = date_pair(expected, actual, ticker, "raw_to_adjusted_ohlc")
        date_rows.append(date_row)
        for field in ["Open", "High", "Low", "Close"]:
            summary, details = compare_numeric_values(
                ticker, field, joined["Date"], joined[f"{field}_expected"],
                joined[f"{field}_actual"], "raw_to_adjusted_ohlc")
            rows.append(summary)
            detail_frames.append(details)
    summary = pd.DataFrame(rows)
    details = pd.concat(detail_frames, ignore_index=True) if detail_frames else pd.DataFrame()
    dates = pd.DataFrame(date_rows)
    write_report("adjusted_ohlc_consistency.csv", summary)
    write_report("adjusted_ohlc_difference_details.csv", details)
    write_report("adjusted_ohlc_date_comparison.csv", dates)
    return summary, details, dates


### Source reconciliation

Compare daily fields and corporate actions by exchange-local date. Use event payload dates; report out-of-range metadata separately.


In [ ]:
def check_json_csv(saved):
    field_rows, field_details, date_rows, events, event_rows, event_details = [], [], [], [], [], []
    event_map = {"dividends": "Dividends", "splits": "Stock Splits", "capitalGains": "Capital Gains"}
    for ticker in ETFS:
        json_name = etf_path(ticker, "yahoo_chart.json").name
        csv_name = etf_path(ticker, "yfinance_raw.csv").name
        if json_name not in saved or csv_name not in saved:
            continue
        native, library = saved[json_name].copy(), saved[csv_name].copy()
        fields = ["Open", "High", "Low", "Close", "Volume", "Adj Close"]
        joined, date_row = date_pair(native[["Date"] + fields], library[["Date"] + fields], ticker, "json_vs_csv")
        date_rows.append(date_row)
        for field in fields:
            summary, details = compare_numeric_values(
                ticker, field, joined["Date"], joined[f"{field}_expected"], joined[f"{field}_actual"], "json_vs_csv")
            field_rows.append(summary)
            field_details.append(details)

        result = json.loads((RAW / json_name).read_bytes())["chart"]["result"][0]
        tz = result["meta"].get("exchangeTimezoneName", "America/New_York")
        ticker_events = []
        for event_type, source_field in event_map.items():
            for key, event in result.get("events", {}).get(event_type, {}).items():
                payload_day = pd.Timestamp(event["date"], unit="s", tz="UTC").tz_convert(tz).tz_localize(None).normalize()
                key_day = pd.Timestamp(int(key), unit="s", tz="UTC").tz_convert(tz).tz_localize(None).normalize()
                if event_type == "splits":
                    numerator, denominator = event.get("numerator"), event.get("denominator")
                    value = numerator / denominator if numerator is not None and denominator not in (None, 0) else np.nan
                else:
                    value = event.get("amount")
                row = {"asset": ticker, "event_type": event_type, "field": source_field,
                       "json_key": key, "json_key_date": key_day, "payload_date": payload_day,
                       "key_payload_date_differ": bool(key_day != payload_day), "json_value": value,
                       "in_requested_range": bool(pd.Timestamp(START) <= payload_day <= pd.Timestamp(END_INCLUSIVE))}
                events.append(row)
                ticker_events.append(row)
        library_index = library.set_index("Date")
        for event_type, source_field in event_map.items():
            selected = [e for e in ticker_events if e["event_type"] == event_type and e["in_requested_range"]]
            by_day = {}
            for event in selected:
                by_day.setdefault(event["payload_date"], []).append(event["json_value"])
            csv_events = numeric(library, source_field) if source_field in library else pd.Series(0.0, index=library.index)
            csv_event_dates = set(library.loc[csv_events.notna() & csv_events.ne(0), "Date"])
            csv_null_dates = set(library.loc[csv_events.isna(), "Date"]) if source_field in library else set()
            compare_days = sorted(set(by_day) | csv_event_dates | csv_null_dates)
            mismatches = duplicates = 0
            for day in compare_days:
                values = by_day.get(day, [])
                if len(values) > 1:
                    duplicates += 1
                    event_details.append({"asset": ticker, "field": source_field, "date": day,
                                          "json_value": str(values), "csv_value": "not_compared",
                                          "issue": "multiple_json_events_same_day; no aggregation performed"})
                    continue
                expected = values[0] if values else 0.0
                actual = (library_index.loc[day, source_field] if source_field in library_index and day in library_index.index
                          else (0.0 if not values else np.nan))
                expected_num = pd.to_numeric(pd.Series([expected]), errors="coerce").iloc[0]
                actual_num = pd.to_numeric(pd.Series([actual]), errors="coerce").iloc[0]
                if not np.isclose(expected_num, actual_num, rtol=COMPARE_RTOL, atol=COMPARE_ATOL, equal_nan=True):
                    mismatches += 1
                    event_details.append({"asset": ticker, "field": source_field, "date": day,
                                          "json_value": expected, "csv_value": actual,
                                          "issue": "event_value_or_date_mismatch"})
            event_rows.append({"asset": ticker, "field": source_field,
                               "json_in_range_events": len(selected), "csv_nonzero_events": len(csv_event_dates),
                               "compared_event_dates": len(compare_days), "mismatched_events": mismatches,
                               "duplicate_event_dates": duplicates,
                               "csv_column_present": source_field in library,
                               "csv_null_rows": len(csv_null_dates)})
    field_df = pd.DataFrame(field_rows)
    details_df = pd.concat(field_details, ignore_index=True) if field_details else pd.DataFrame()
    dates_df = pd.DataFrame(date_rows)
    events_df = pd.DataFrame(events)
    outside_df = events_df.loc[~events_df["in_requested_range"]].copy() if not events_df.empty else pd.DataFrame()
    event_df = pd.DataFrame(event_rows)
    event_detail_df = pd.DataFrame(event_details, columns=["asset", "field", "date", "json_value", "csv_value", "issue"])
    for filename, frame in [("json_csv_field_comparison.csv", field_df),
                            ("json_csv_field_difference_details.csv", details_df),
                            ("json_csv_date_comparison.csv", dates_df),
                            ("json_corporate_actions.csv", events_df),
                            ("json_out_of_range_events.csv", outside_df),
                            ("json_csv_event_comparison.csv", event_df),
                            ("json_csv_event_difference_details.csv", event_detail_df)]:
        write_report(filename, frame)
    return {"json_csv_fields": field_df, "json_csv_details": details_df, "json_csv_dates": dates_df,
            "json_events": events_df, "json_outside_events": outside_df,
            "json_csv_events": event_df, "json_csv_event_details": event_detail_df}


### Opening gaps

Report distributions and annual summaries of `Open / previous Close - 1`. A near-zero threshold of 1e-6 is diagnostic only.


In [ ]:
def check_opening_gaps(saved):
    near_zero_threshold = 1e-6
    summaries, yearly, extremes = [], [], []
    def describe(part, ticker, version, year="ALL"):
        valid = part.loc[part["gap"].notna() & np.isfinite(part["gap"])].copy()
        quantiles = valid["gap"].quantile([0, 0.01, 0.05, 0.5, 0.95, 0.99, 1])
        row = {"asset": ticker, "version": version, "year": year, "rows": len(part),
               "valid_gap_rows": len(valid), "unavailable_gap_rows": len(part)-len(valid),
               "exact_zero_gap_rows": int(valid["gap"].eq(0).sum()),
               "near_zero_gap_rows": int(valid["gap"].abs().le(near_zero_threshold).sum()),
               "near_zero_threshold": near_zero_threshold,
               "repeated_open_rows": int(part["repeated_open"].sum()),
               "comparable_open_pairs": int(part["previous_open"].notna().sum()),
               "max_absolute_gap": valid["gap"].abs().max()}
        for label, q in [("min", 0), ("p01", 0.01), ("p05", 0.05), ("median", 0.5), ("p95", 0.95), ("p99", 0.99), ("max", 1)]:
            row[label] = quantiles.loc[q]
        row["near_zero_gap_fraction"] = row["near_zero_gap_rows"] / len(valid) if len(valid) else np.nan
        row["repeated_open_fraction"] = row["repeated_open_rows"] / row["comparable_open_pairs"] if row["comparable_open_pairs"] else np.nan
        return row
    for ticker in ETFS:
        for suffix, version in [("yfinance_raw.csv", "raw_returned"), ("yfinance_adjusted.csv", "auto_adjusted")]:
            filename = etf_path(ticker, suffix).name
            if filename not in saved:
                continue
            f = saved[filename].copy().sort_values("Date").reset_index(drop=True)
            opened, closed = numeric(f, "Open"), numeric(f, "Close")
            previous_close, previous_open = closed.shift(1), opened.shift(1)
            diagnostics = pd.DataFrame({"asset": ticker, "version": version, "date": f["Date"],
                                        "previous_date": f["Date"].shift(1), "Open": opened,
                                        "previous_close": previous_close, "previous_open": previous_open,
                                        "gap": opened / previous_close - 1,
                                        "repeated_open": opened.eq(previous_open) & previous_open.notna(),
                                        "Dividends": numeric(f, "Dividends") if "Dividends" in f else np.nan,
                                        "Stock Splits": numeric(f, "Stock Splits") if "Stock Splits" in f else np.nan})
            summaries.append(describe(diagnostics, ticker, version))
            for year, group in diagnostics.groupby(diagnostics["date"].dt.year):
                yearly.append(describe(group, ticker, version, int(year)))
            diagnostics["absolute_gap"] = diagnostics["gap"].abs()
            extremes.append(diagnostics.nlargest(10, "absolute_gap"))
    overall, annual = pd.DataFrame(summaries), pd.DataFrame(yearly)
    extreme = pd.concat(extremes, ignore_index=True) if extremes else pd.DataFrame()
    write_report("opening_gap_summary.csv", overall)
    write_report("opening_gap_by_year.csv", annual)
    write_report("opening_gap_extreme_dates.csv", extreme)
    return overall, annual, extreme


In [ ]:
def json_frame(path):
    result = json.loads(path.read_bytes())["chart"]["result"][0]
    tz = result["meta"].get("exchangeTimezoneName", "America/New_York")
    timestamps = result.get("timestamp", [])
    dates = pd.to_datetime(timestamps, unit="s", utc=True).tz_convert(tz).tz_localize(None).normalize()
    quote = result["indicators"]["quote"][0]
    names = {"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"}
    frame = pd.DataFrame({names[k]: quote.get(k, [None] * len(dates)) for k in names})
    adj = result["indicators"].get("adjclose", [{}])[0].get("adjclose")
    if adj is not None:
        frame["Adj Close"] = adj
    frame.insert(0, "Date", dates)
    return frame

def read_saved(path):
    if path.suffix == ".json":
        return json_frame(path)
    frame = pd.read_csv(path, keep_default_na=False)
    col = next(c for c in ["Date", "DATE", "observation_date"] if c in frame.columns)
    raw_dates = frame[col].astype(str)
    if path == VIX_PATH:
        dates = pd.to_datetime(raw_dates, format="%m/%d/%Y", errors="raise")
    elif "yfinance" in path.name:
        # Preserve the exchange-local date across daylight-saving offsets.
        dates = pd.to_datetime(raw_dates.str.slice(0, 10), format="%Y-%m-%d", errors="raise")
    else:
        dates = pd.to_datetime(raw_dates, format="%Y-%m-%d", errors="raise")
    frame = frame.rename(columns={col: "Date", "OPEN": "Open", "HIGH": "High", "LOW": "Low", "CLOSE": "Close"})
    frame["Date"] = dates
    return frame

def numeric(frame, column):
    return pd.to_numeric(frame[column], errors="coerce")

def write_report(name, frame):
    frame.to_csv(SOURCE_REPORTS / name, index=False)

def run_quality_checks():
    checks_before = {p.name: sha256_bytes(p.read_bytes()) for p in expected_paths if p.exists()}
    saved = {}
    summary = []
    findings = []
    price_columns = ["Open", "High", "Low", "Close", "Adj Close"]
    null_details = []
    price_bad_rows = []
    for path in expected_paths:
        if not path.exists():
            findings.append({"level": "ERROR", "asset": path.name.split('_')[0], "check": "missing_file", "detail": path.name})
            continue
        frame = read_saved(path)
        entry = download_manifest["files"][path.name]
        asset, kind = entry["asset"], entry["kind"]
        saved[path.name] = frame
        dates = frame["Date"]
        duplicate_dates = int(dates.duplicated().sum())
        outside = int(((dates < START) | (dates > END_INCLUSIVE)).sum())
        row = {"asset": asset, "version": kind, "file": path.name,
               "start": str(dates.min().date()), "end": str(dates.max().date()), "rows": len(frame),
               "duplicate_dates": duplicate_dates, "dates_increasing": bool(dates.is_monotonic_increasing),
               "outside_requested_range": outside, "open_outside_low_high": pd.NA,
               "open_check_unavailable": pd.NA, "price_zero_cells": pd.NA,
               "price_null_cells": pd.NA, "volume_zero_rows": pd.NA, "volume_null_rows": pd.NA,
               "rf_zero_rows": pd.NA, "rf_null_rows": pd.NA}
        present = [c for c in price_columns if c in frame]
        if present:
            prices = frame[present].apply(pd.to_numeric, errors="coerce")
            row["price_zero_cells"] = int(prices.eq(0).sum().sum())
            row["price_null_cells"] = int(prices.isna().sum().sum())
            for c in present:
                null_details.append({"asset": asset, "version": kind, "field": c,
                                     "null_count": int(prices[c].isna().sum()),
                                     "zero_count": int(prices[c].eq(0).sum())})
            o, h, l = (numeric(frame, c) for c in ["Open", "High", "Low"])
            valid = pd.Series(np.isfinite(o) & np.isfinite(h) & np.isfinite(l), index=frame.index)
            tolerance = 1e-8 * pd.concat([o.abs(), h.abs(), l.abs()], axis=1).max(axis=1).clip(lower=1)
            bad = valid & ((o < l - tolerance) | (o > h + tolerance) | (h < l - tolerance))
            row["open_outside_low_high"] = int(bad.sum())
            row["open_check_unavailable"] = int((~valid).sum())
            if bad.any():
                for _, values in frame.loc[bad, ["Date", "Open", "High", "Low", "Close"]].iterrows():
                    price_bad_rows.append({"asset": asset, "version": kind, **values.to_dict()})
            if bad.any() or row["price_zero_cells"] or row["price_null_cells"] or prices.lt(0).any().any():
                findings.append({"level": "REVIEW", "asset": asset, "check": "price_fields",
                                 "detail": f"{kind}: Open outside range={int(bad.sum())}, zero prices={row['price_zero_cells']}, missing prices={row['price_null_cells']}"})
        if "Volume" in frame:
            volume = numeric(frame, "Volume")
            row["volume_zero_rows"] = int(volume.eq(0).sum())
            row["volume_null_rows"] = int(volume.isna().sum())
            null_details.append({"asset": asset, "version": kind, "field": "Volume",
                                 "null_count": int(volume.isna().sum()), "zero_count": int(volume.eq(0).sum())})
            if volume.isna().any() or volume.eq(0).any() or volume.lt(0).any():
                findings.append({"level": "REVIEW", "asset": asset, "check": "volume_fields",
                                 "detail": f"{kind}: zero volume={int(volume.eq(0).sum())}, missing volume={int(volume.isna().sum())}"})
        if "DGS3MO" in frame:
            rates = numeric(frame, "DGS3MO")
            row["rf_zero_rows"], row["rf_null_rows"] = int(rates.eq(0).sum()), int(rates.isna().sum())
            findings.append({"level": "INFO", "asset": asset, "check": "rate_missing_markers",
                             "detail": f"Missing rate observations: {int(rates.isna().sum())}; valid zero rates: {int(rates.eq(0).sum())}."})
        if duplicate_dates or outside or not dates.is_monotonic_increasing:
            findings.append({"level": "REVIEW", "asset": asset, "check": "date_integrity", "detail": str(row)})
        summary.append(row)
    summary_df = pd.DataFrame(summary)
    write_report("file_summary.csv", summary_df)
    write_report("field_null_zero_details.csv", pd.DataFrame(null_details))
    write_report("price_range_violations.csv", pd.DataFrame(price_bad_rows, columns=["asset", "version", "Date", "Open", "High", "Low", "Close"]))

    spy = saved.get(etf_path("SPY", "yfinance_raw.csv").name)
    if spy is None:
        raise RuntimeError("The raw SPY CSV is required for calendar checks.")
    spy_dates = set(spy["Date"])
    calendar_rows, difference_rows = [], []
    for filename, frame in saved.items():
        entry = download_manifest["files"][filename]
        dates = set(frame["Date"])
        missing = sorted(spy_dates - dates)
        extra = sorted(dates - spy_dates)
        first, last = frame["Date"].min(), frame["Date"].max()
        before = [d for d in missing if d < first]
        inside = [d for d in missing if first <= d <= last]
        after = [d for d in missing if d > last]
        non_quote = 0
        if entry["asset"] == "DGS3MO":
            non_quote = int((frame["Date"].isin(spy_dates) & numeric(frame, "DGS3MO").isna()).sum())
        calendar_rows.append({"asset": entry["asset"], "version": entry["kind"], "file": filename,
                              "missing_vs_spy": len(missing), "before_first_observation": len(before),
                              "within_observed_span": len(inside), "after_last_observation": len(after),
                              "extra_vs_spy": len(extra), "spy_dates_with_missing_rate": non_quote})
        for category, days in [("before_first_observation", before), ("within_observed_span", inside),
                               ("after_last_observation", after), ("extra_vs_spy", extra)]:
            difference_rows.extend({"asset": entry["asset"], "version": entry["kind"],
                                    "file": filename, "category": category, "date": d.date().isoformat()} for d in days)
        if entry["asset"] in ETFS and (inside or after or extra):
            findings.append({"level": "REVIEW", "asset": entry["asset"], "check": "calendar_difference",
                             "detail": f"{entry['kind']}: missing within observed span={len(inside)}, missing after final observation={len(after)}, additional dates={len(extra)}"})
    calendar_df = pd.DataFrame(calendar_rows)
    write_report("calendar_summary.csv", calendar_df)
    write_report("calendar_difference_dates.csv", pd.DataFrame(difference_rows, columns=["asset", "version", "file", "category", "date"]))

    adj_ohlc, adj_details, adj_dates = check_adjusted_ohlc(saved)
    matched_dates = {r.asset: r.left_rows-r.left_only_dates for r in adj_dates.itertuples()}
    crosschecks = [{"asset": r.asset, "matched_rows": matched_dates[r.asset],
                    "max_adjusted_close_difference": r.max_abs_difference}
                   for r in adj_ohlc.loc[adj_ohlc["field"].eq("Close")].itertuples()]
    top_frames, dividend_candidates, split_candidates = [], [], []
    raw_frames = {}
    for ticker in ETFS:
        path = etf_path(ticker, "yfinance_raw.csv")
        if path.name not in saved:
            continue
        frame = saved[path.name].copy().sort_values("Date").reset_index(drop=True)
        raw_frames[ticker] = frame
        close, adj = numeric(frame, "Close"), numeric(frame, "Adj Close")
        close_return = close.pct_change(fill_method=None)
        adj_return = adj.pct_change(fill_method=None)
        div = numeric(frame, "Dividends") if "Dividends" in frame else pd.Series(np.nan, index=frame.index)
        split = numeric(frame, "Stock Splits") if "Stock Splits" in frame else pd.Series(np.nan, index=frame.index)
        table = pd.DataFrame({"asset": ticker, "date": frame["Date"], "Close": close,
                              "Adj Close": adj, "close_return": close_return,
                              "adj_close_return": adj_return, "abs_adj_close_return": adj_return.abs(),
                              "Dividends": div, "Stock Splits": split})
        top_frames.append(table.nlargest(10, "abs_adj_close_return"))
        for i in frame.index[(div > 0) & (split.isna() | split.eq(0))]:
            if i > 0 and pd.notna(close.iloc[i-1]) and close.iloc[i-1] > 0 and pd.notna(adj.iloc[i-1]):
                dividend_candidates.append((ticker, int(i)))
        for i in frame.index[(split > 0) & (split != 1)]:
            if i > 0:
                split_candidates.append((frame.loc[i, "Date"], ticker, int(i), float(split.iloc[i])))
    top_df = pd.concat(top_frames, ignore_index=True) if top_frames else pd.DataFrame()
    write_report("etf_top10_absolute_daily_returns.csv", top_df)
    for item in crosschecks:
        if item["matched_rows"] != len(raw_frames[item["asset"]]) or item["max_adjusted_close_difference"] > 1e-8:
            findings.append({"level": "REVIEW", "asset": item["asset"],
                             "check": "two_download_versions", "detail": str(item)})
    write_report("adjusted_version_crosscheck.csv", pd.DataFrame(crosschecks))

    dividend_df = pd.DataFrame()
    if dividend_candidates:
        rng = np.random.default_rng(RANDOM_SEED)
        ticker, i = dividend_candidates[int(rng.integers(len(dividend_candidates)))]
        f = raw_frames[ticker]
        p0, p1, d = float(f.loc[i-1, "Close"]), float(f.loc[i, "Close"]), float(f.loc[i, "Dividends"])
        a0, a1 = float(f.loc[i-1, "Adj Close"]), float(f.loc[i, "Adj Close"])
        manual = (p1 + d) / p0 - 1
        vendor = a1 / a0 - 1
        dividend_df = pd.DataFrame([{"asset": ticker, "date": f.loc[i, "Date"],
                                     "previous_date": f.loc[i-1, "Date"], "previous_close": p0,
                                     "close": p1, "dividend": d, "previous_adj_close": a0,
                                     "adj_close": a1, "cash_dividend_return": manual,
                                     "adj_close_return": vendor, "difference_bps": (manual-vendor)*10000,
                                     "seed": RANDOM_SEED}])
        adjusted_sample = saved[etf_path(ticker, "yfinance_adjusted.csv").name].set_index("Date")
        entry_day, exit_day = f.loc[i-1, "Date"], f.loc[i, "Date"]
        open_entry, open_exit = float(f.loc[i-1, "Open"]), float(f.loc[i, "Open"])
        adj_open_entry, adj_open_exit = float(adjusted_sample.loc[entry_day, "Open"]), float(adjusted_sample.loc[exit_day, "Open"])
        open_cash_return = (open_exit + d) / open_entry - 1
        open_adjusted_return = adj_open_exit / adj_open_entry - 1
        for key, value in {"previous_open": open_entry, "open": open_exit,
                           "previous_adjusted_open": adj_open_entry, "adjusted_open": adj_open_exit,
                           "open_cash_dividend_return": open_cash_return,
                           "adjusted_open_return": open_adjusted_return,
                           "open_return_difference_bps": (open_cash_return-open_adjusted_return)*10000}.items():
            dividend_df[key] = value
        findings.append({"level": "INFO", "asset": ticker, "check": "dividend_check",
                         "detail": f"{f.loc[i, 'Date'].date()}: cash-minus-adjusted return = {(manual-vendor)*10000:.8f} bps."})
    else:
        findings.append({"level": "REVIEW", "asset": "ALL", "check": "dividend_check", "detail": "No dividend event with an available previous close."})
    write_report("random_dividend_check.csv", dividend_df)

    split_window, split_result = pd.DataFrame(), pd.DataFrame()
    all_split_events = pd.DataFrame([
        {"asset": ticker, "date": day, "split_ratio": ratio,
         "classification": "ordinary_2_for_1" if np.isclose(ratio, 2.0) else "other_ratio_review"}
        for day, ticker, i, ratio in sorted(split_candidates)
    ])
    write_report("all_split_events.csv", all_split_events)
    for day, ticker, i, ratio in split_candidates:
        if not np.isclose(ratio, 2.0):
            findings.append({"level": "REVIEW", "asset": ticker, "check": "nonstandard_corporate_action",
                             "detail": f"{day.date()}: vendor Stock Splits={ratio:g}; a nonstandard corporate action requires separate interpretation."})
    if split_candidates:
        day, ticker, i, ratio = sorted(split_candidates,
                                      key=lambda item: (not np.isclose(item[3], 2.0), item[0], item[1]))[0]
        f = raw_frames[ticker]
        window = f.iloc[max(0, i-3):i+4].copy()
        split_window = pd.DataFrame({"asset": ticker, "date": window["Date"],
                                     "Close": numeric(window, "Close"), "Volume": numeric(window, "Volume"),
                                     "Stock Splits": numeric(window, "Stock Splits")})
        split_window["Close_x_Volume"] = split_window["Close"] * split_window["Volume"]
        split_window["is_split_date"] = split_window["date"].eq(day)
        close = numeric(f, "Close")
        volume = numeric(f, "Volume")
        dv = close * volume
        before = dv.iloc[max(0, i-3):i].median()
        after = dv.iloc[i:i+3].median()
        split_result = pd.DataFrame([{"asset": ticker, "date": day, "split_ratio": ratio,
                                     "close_event_over_previous": close.iloc[i]/close.iloc[i-1],
                                     "volume_event_over_previous": volume.iloc[i]/volume.iloc[i-1],
                                     "dollar_volume_event_over_previous": dv.iloc[i]/dv.iloc[i-1],
                                     "median_dollar_volume_3day_after_over_before": after/before,
                                     "note": "Dollar volume varies naturally; prices may already be split-adjusted. Ratios are descriptive."}])
        findings.append({"level": "INFO", "asset": ticker, "check": "split_units_review",
                         "detail": f"{day.date()} split ratio {ratio:g}; inspect price and volume units around the event."})
    else:
        findings.append({"level": "INFO", "asset": "ALL", "check": "split_check_unavailable",
                         "detail": "No split event is available in the selected universe and period."})
    write_report("split_event_window.csv", split_window)
    write_report("split_event_summary.csv", split_result)
    reconciliation = check_json_csv(saved)
    gap_summary, gap_yearly, gap_extremes = check_opening_gaps(saved)
    for date_table in [adj_dates, reconciliation["json_csv_dates"]]:
        for _, row in date_table.iterrows():
            if row["left_only_dates"] or row["right_only_dates"]:
                findings.append({"level": "REVIEW", "asset": row["asset"], "check": "paired_date_difference",
                                 "detail": f"{row['comparison']}: left-only dates={int(row['left_only_dates'])}, right-only dates={int(row['right_only_dates'])}."})
    for _, result in adj_ohlc.loc[adj_ohlc["different_rows"].gt(0)].iterrows():
        findings.append({"level": "REVIEW", "asset": result["asset"], "check": "adjusted_ohlc_conversion",
                         "detail": f"{result['field']}: {int(result['different_rows'])} rows exceed tolerance; maximum difference={result['max_abs_difference']:.12g}."})
    for _, result in reconciliation["json_csv_fields"].loc[lambda f: f["different_rows"].gt(0)].iterrows():
        findings.append({"level": "REVIEW", "asset": result["asset"], "check": "json_csv_field_difference",
                         "detail": f"{result['field']}: {int(result['different_rows'])} rows exceed tolerance; maximum difference={result['max_abs_difference']:.12g}, null-mask mismatches={int(result['null_mask_mismatches'])}; independent response differences retained."})
    for _, result in reconciliation["json_csv_events"].loc[lambda f: (f["mismatched_events"] + f["duplicate_event_dates"]).gt(0)].iterrows():
        findings.append({"level": "REVIEW", "asset": result["asset"], "check": "json_csv_event_difference",
                         "detail": f"{result['field']}: mismatched events={int(result['mismatched_events'])}, duplicate event dates={int(result['duplicate_event_dates'])}."})
    for _, event in reconciliation["json_outside_events"].iterrows():
        findings.append({"level": "INFO", "asset": event["asset"], "check": "json_out_of_range_metadata",
                         "detail": f"{event['event_type']} payload date={event['payload_date'].date()}, dictionary-key date={event['json_key_date'].date()}; excluded from in-range event reconciliation."})
    rates = saved.get(FRED_PATH.name)
    zero_rate_years = pd.DataFrame()
    if rates is not None:
        zero_rows = rates.loc[numeric(rates, "DGS3MO").eq(0), ["Date", "DGS3MO"]].copy()
        zero_rate_years = zero_rows.groupby(zero_rows["Date"].dt.year).size().rename("zero_rate_rows").reset_index().rename(columns={"Date": "year"})
        write_report("risk_free_zero_rate_dates.csv", zero_rows)
        write_report("risk_free_zero_rate_years.csv", zero_rate_years)
    write_report("findings.csv", pd.DataFrame(findings, columns=["level", "asset", "check", "detail"]))

    checks_after = {p.name: sha256_bytes(p.read_bytes()) for p in expected_paths if p.exists()}
    changed = [name for name, digest in checks_before.items() if checks_after.get(name) != digest]
    manifest_mismatch = [name for name, digest in checks_after.items()
                         if digest != download_manifest["files"][name]["sha256"]]
    if changed or manifest_mismatch:
        raise AssertionError(f"Raw snapshot integrity check failed: {changed}, {manifest_mismatch}")
    return {"files": summary_df, "calendar": calendar_df, "top10": top_df,
            "dividend": dividend_df, "split_window": split_window, "split_summary": split_result,
            "findings": pd.DataFrame(findings), "crosschecks": pd.DataFrame(crosschecks),
            "all_split_events": all_split_events,
            "price_violations": pd.DataFrame(price_bad_rows),
            "integrity_verified": True, "missing_files": missing_files,
            "adjusted_ohlc": adj_ohlc, "adjusted_ohlc_details": adj_details,
            "adjusted_ohlc_dates": adj_dates, **reconciliation,
            "gap_summary": gap_summary, "gap_yearly": gap_yearly, "gap_extremes": gap_extremes,
            "zero_rate_years": zero_rate_years}


In [ ]:
qa = run_quality_checks()
with pd.option_context("display.max_rows", 160, "display.max_columns", 30,
                       "display.width", 180, "display.float_format", lambda x: f"{x:.10g}"):
    for title, key in [
        ("File coverage and OHLC checks", "files"),
        ("Calendar differences relative to SPY", "calendar"),
        ("Ten largest absolute daily returns per ETF", "top10"),
        ("Dividend-event comparison", "dividend"),
        ("Split-event price and volume", "split_window"),
        ("Split-event summary", "split_summary"),
        ("Data-quality findings", "findings"),
    ]:
        print(title)
        display(qa[key])
print(f"Snapshots: {len(qa['files'])}/{len(expected_paths)}; integrity verified: {qa['integrity_verified']}")
print("Diagnostic reports:", SOURCE_REPORTS.relative_to(ROOT))


## Cleaning and calendar alignment

### Return conventions

Cash distributions accrue on the ex-date. Supplier OHLC and volume are already split-normalized.

$r^{CC}_t=(C_t+D_t)/C_{t-1}-1$, $r^{OO}_t=(O_t+D_t)/O_{t-1}-1$.

### Availability

Market observations are available at 18:00 New York time. Treasury yields use publication-calendar-aware as-of matching.

In [ ]:
steps, changes, calendar_changes = [], [], []

def log_step(stage, asset, field, action, count, reason):
    steps.append(dict(stage=stage, asset=asset, field=field, action=action,
                      affected_rows=int(count), reason=reason))

def record_changes(asset, frame, field, mask, replacement, rule):
    for idx in frame.index[mask]:
        changes.append(dict(asset=asset, date=frame.at[idx, "date"],
                            field=field, original_value=repr(frame.at[idx, field]),
                            cleaned_value=replacement, rule=rule))

download_manifest_hash = sha256(DOWNLOAD_MANIFEST_PATH)
download_manifest = json.loads(DOWNLOAD_MANIFEST_PATH.read_text(encoding='utf-8'))
expected_hashes = {name: meta["sha256"]
                   for name, meta in download_manifest["files"].items()}
actual_hashes = {name: sha256(RAW / name) for name in expected_hashes}
assert len(expected_hashes) == 47
assert actual_hashes == expected_hashes, "Raw files do not match the download manifest."
print(f"{ROOT.name}: 47 raw file hashes verified; {RULE_VERSION}.")


### Dates and types

In [ ]:
def normalise_dates(frame, date_column, asset, exchange_timestamp=False):
    out = frame.copy()
    if exchange_timestamp:
        parsed = pd.to_datetime(out[date_column], utc=True, errors="raise")
        dates = parsed.dt.tz_convert(NY).dt.tz_localize(None).dt.normalize()
        assert dates.dt.strftime("%Y-%m-%d").eq(
            out[date_column].astype(str).str[:10]).all()
    else:
        dates = pd.to_datetime(out[date_column], errors="raise").dt.normalize()
    out = out.rename(columns={date_column: "date"})
    out["date"] = dates
    if not out["date"].between(START_DATE, END_DATE).all():
        raise ValueError(f"{asset}: dates outside the study period.")
    duplicate = out.duplicated("date", keep=False)
    for date, group in out.loc[duplicate].groupby("date"):
        if len(group.drop_duplicates()) > 1:
            raise ValueError(f"{asset}: conflicting duplicate records on {date.date()}.")
    removed = len(out) - len(out.drop_duplicates("date"))
    out = out.drop_duplicates("date").sort_values("date").reset_index(drop=True)
    log_step("date", asset, "date", "normalise_sort_deduplicate", removed,
             "New York trading dates; remove identical duplicates and reject conflicts.")
    return out

def numeric_columns(frame, columns, asset):
    out = frame.copy()
    for field in columns:
        values = pd.to_numeric(out[field], errors="raise").astype("float64")
        if np.isinf(values.to_numpy()).any():
            raise ValueError(f"{asset} {field}: infinite values.")
        out[field] = values
    log_step("type", asset, ",".join(columns), "numeric_float64", len(out),
             "Parse float64 values without scaling or rounding.")
    return out

def isolate_bad_ohlc(frame, asset):
    out = frame.copy()
    reasons = pd.Series("", index=out.index, dtype="str")
    for field in ["open", "high", "low", "close"]:
        bad = out[field].notna() & (out[field] <= 0)
        record_changes(asset, out, field, bad, None, "nonpositive_price")
        out.loc[bad, field] = np.nan
        reasons.loc[bad] += f"{field}_nonpositive;"
    inverted = out["high"].notna() & out["low"].notna() & (out["high"] < out["low"])
    for field in ["high", "low"]:
        record_changes(asset, out, field, inverted, None, "high_below_low")
        out.loc[inverted, field] = np.nan
    reasons.loc[inverted] += "high_below_low;"
    for field in ["open", "close"]:
        bad = (out[[field, "low", "high"]].notna().all(axis=1)
               & ((out[field] < out["low"]) | (out[field] > out["high"])))
        record_changes(asset, out, field, bad, None, f"{field}_outside_low_high")
        out.loc[bad, field] = np.nan
        reasons.loc[bad] += f"{field}_outside_low_high;"
    out["invalid_reason"] = reasons.str.rstrip(";")
    log_step("price", asset, "OHLC", "isolate_invalid_fields",
             reasons.ne("").sum(), "Isolate invalid fields; retain dates and genuine extreme returns.")
    return out


### ETF alignment

In [ ]:
field_map = {
    "Open": "open", "High": "high", "Low": "low", "Close": "close",
    "Adj Close": "adj_close", "Volume": "volume", "Dividends": "dividends",
    "Stock Splits": "stock_splits", "Capital Gains": "capital_gains",
}
etf_native, etf_aligned, native_rows = {}, {}, {}
for ticker in ETFS:
    path = RAW / f"{ticker}_{SUFFIX}_yfinance_raw.csv"
    raw = pd.read_csv(path)
    native_rows[ticker] = len(raw)
    data = normalise_dates(raw, "Date", ticker, exchange_timestamp=True)
    data = data.rename(columns=field_map)
    required = list(field_map.values())
    if not set(required).issubset(data.columns):
        raise ValueError(f"{ticker}: missing required fields.")
    data = numeric_columns(data, required, ticker)
    data = isolate_bad_ohlc(data, ticker)
    bad_volume = data["volume"].notna() & (data["volume"] < 0)
    record_changes(ticker, data, "volume", bad_volume, None, "negative_volume")
    data.loc[bad_volume, "volume"] = np.nan
    data.loc[bad_volume, "invalid_reason"] += ";negative_volume"
    if (data["dividends"].dropna() < 0).any():
        raise ValueError(f"{ticker}: negative dividends require review.")
    if data["capital_gains"].fillna(0).ne(0).any():
        raise ValueError(f"{ticker}: nonzero capital-gain distributions require review.")
    if not np.equal(data["volume"].dropna(), np.floor(data["volume"].dropna())).all():
        raise ValueError(f"{ticker}: noninteger volume.")
    data["volume"] = data["volume"].astype("Int64")
    bad_factor = data["adj_close"].notna() & (data["adj_close"] <= 0)
    if bad_factor.any():
        raise ValueError(f"{ticker}: invalid adjusted close.")
    data["adjustment_factor"] = data["adj_close"] / data["close"]
    for field in ["open", "high", "low", "close"]:
        data[f"adjusted_{field}"] = data[field] * data["adjustment_factor"]
    data["special_action"] = (ticker == "XLF") & data["date"].eq(XLF_EVENT)
    data.loc[data["special_action"], "invalid_reason"] = (
        data.loc[data["special_action"], "invalid_reason"]
        .apply(lambda s: ";".join(filter(None, [s, "xlf_in_kind_distribution"]))))
    data["volume_zero"] = data["volume"].eq(0).fillna(False)
    data["ticker"] = ticker
    etf_native[ticker] = data

spy_calendar = pd.DatetimeIndex(etf_native["SPY"]["date"], name="date")
assert spy_calendar.is_unique and spy_calendar.is_monotonic_increasing
assert len(spy_calendar) == 5218
for ticker in ETFS:
    native = etf_native[ticker].set_index("date")
    extra = native.index.difference(spy_calendar)
    if len(extra):
        raise ValueError(f"{ticker}: observations outside the SPY calendar require review.")
    aligned = native.reindex(spy_calendar).reset_index()
    aligned["ticker"] = ticker
    aligned["observed"] = aligned["date"].isin(native.index)
    before = aligned["date"] < native.index.min()
    inside = ~aligned["observed"] & ~before
    aligned["invalid_reason"] = aligned["invalid_reason"].fillna("")
    aligned.loc[before, "invalid_reason"] = "before_first_observation"
    aligned.loc[inside, "invalid_reason"] = "missing_observation"
    for field in ["special_action", "volume_zero"]:
        aligned[field] = aligned[field].fillna(False).astype(bool)
    aligned["ohlc_valid"] = aligned[["open", "high", "low", "close"]].notna().all(axis=1)
    aligned["quote_available_at"] = (
        (aligned["date"] + pd.Timedelta(hours=18)).dt.tz_localize(NY).astype(str))
    aligned.loc[~aligned["observed"], "quote_available_at"] = None
    for date in aligned.loc[~aligned["observed"], "date"]:
        calendar_changes.append(dict(asset=ticker, date=date, action="add_empty_calendar_row",
                                     reason="before_first_observation" if date < native.index.min()
                                     else "missing_within_coverage"))
    log_step("calendar", ticker, "all", "reindex_spy_without_fill",
             (~aligned["observed"]).sum(), "Align to SPY without imputing market observations.")
    etf_aligned[ticker] = aligned.set_index("date")

etf_prices = pd.concat(
    [d.reset_index() for d in etf_aligned.values()], ignore_index=True
).sort_values(["date", "ticker"]).reset_index(drop=True)


### Corporate actions

In [ ]:
dividend_records, action_records = [], []
for ticker, data in etf_aligned.items():
    q = data["adjustment_factor"]
    implied = data["close"].shift(1) * (1 - q.shift(1) / q)
    div_dates = data.index[data["dividends"].fillna(0).ne(0)]
    for date in div_dates:
        actual = data.at[date, "dividends"]
        inferred = implied.loc[date]
        dividend_records.append(dict(
            ticker=ticker, date=date, dividend=actual, implied_dividend=inferred,
            abs_difference=abs(actual - inferred),
            within_unit_tolerance=bool(pd.notna(inferred) and abs(actual-inferred) <= 0.001)))
    for date in data.index[data["stock_splits"].fillna(0).ne(0)]:
        k = spy_calendar.get_loc(date)
        ratio = data.at[date, "stock_splits"]
        special = bool(data.at[date, "special_action"])
        action_records.append(dict(
            ticker=ticker, date=date, reported_split_ratio=ratio,
            classification="in_kind_distribution" if special else "ordinary_split",
            previous_close=data.iloc[k-1]["close"],
            current_open=data.at[date, "open"],
            open_over_previous_close=data.at[date, "open"] / data.iloc[k-1]["close"],
            treatment="cross_event_returns_invalid" if special
                      else "retain_provider_units_no_second_adjustment",
            volume_treatment="retain_provider_volume; dollar_volume_is_proxy"))
dividend_unit_checks = pd.DataFrame(dividend_records)
corporate_action_checks = pd.DataFrame(action_records)
assert len(corporate_action_checks) == 6
assert len(corporate_action_checks.query("classification == 'ordinary_split'")) == 5
if not dividend_unit_checks["within_unit_tolerance"].all():
    raise ValueError("Dividend-unit checks failed; review the diagnostic records.")
display(corporate_action_checks)


## Returns and corporate actions

In [ ]:
return_tables, dividend_return_comparisons = [], []
cross_names = ["ret_cc_cash", "ret_oo_cash", "ret_overnight_cash"]
for ticker, data in etf_aligned.items():
    out = pd.DataFrame(index=spy_calendar)
    out["ticker"] = ticker
    out["previous_date"] = pd.Series(spy_calendar, index=spy_calendar).shift(1)
    distributions_known = data["dividends"].notna() & data["capital_gains"].notna()
    distribution = data["dividends"]
    out["ret_cc_cash"] = (data["close"] + distribution) / data["close"].shift(1) - 1
    out["ret_oo_cash"] = (data["open"] + distribution) / data["open"].shift(1) - 1
    out["ret_overnight_cash"] = (data["open"] + distribution) / data["close"].shift(1) - 1
    out["ret_intraday_price"] = data["close"] / data["open"] - 1
    for name in cross_names:
        out[name] = out[name].where(distributions_known & ~data["special_action"])
        out[f"{name}_valid"] = out[name].notna()
        reason = pd.Series("", index=spy_calendar, dtype="str")
        reason.loc[out[name].isna()] = "missing_endpoint_or_distribution"
        reason.loc[data["special_action"]] = "xlf_in_kind_distribution"
        reason.iloc[0] = "no_previous_spy_date"
        out[f"{name}_invalid_reason"] = reason
    out["ret_intraday_price_valid"] = out["ret_intraday_price"].notna()
    out["invalid_reason"] = data["invalid_reason"]
    return_tables.append(out.reset_index())

    proxy = data["adjusted_open"] / data["adjusted_open"].shift(1) - 1
    for date in data.index[distribution.fillna(0).ne(0) & out["ret_oo_cash_valid"]]:
        dividend_return_comparisons.append(dict(
            ticker=ticker, date=date, cash_open_return=out.at[date, "ret_oo_cash"],
            adjusted_open_proxy_return=proxy.loc[date],
            cash_minus_proxy_bps=(out.at[date, "ret_oo_cash"]-proxy.loc[date])*10000))

base_returns = pd.concat(return_tables, ignore_index=True).sort_values(
    ["date", "ticker"]).reset_index(drop=True)
dividend_return_comparison = pd.DataFrame(dividend_return_comparisons)
returns_by_ticker = {t: base_returns.loc[base_returns["ticker"].eq(t)].set_index("date")
                     for t in ETFS}
display(returns_by_ticker["XLF"].loc["2016-09-15":"2016-09-21",
        ["ret_cc_cash", "ret_oo_cash", "ret_overnight_cash",
         "ret_intraday_price", "ret_oo_cash_invalid_reason"]])


### XLF dependency windows

Exclude only returns and windows that depend on the unverified 2016-09-19 distribution.

In [ ]:
impact_specs = [
    ("own_return_1d", 1, 0, "XLF"),
    ("own_return_5d", 5, 0, "XLF"),
    ("own_return_20d", 20, 0, "XLF"),
    ("own_return_or_volatility_60d", 60, 0, "XLF"),
    ("ordinary_return_252d_for_comparison", 252, 0, "XLF"),
    ("momentum_12_1", 252, 21, "XLF"),
    ("cross_sector_XLF_return_20d", 20, 0, "other_8_sectors_S1"),
    ("nine_sector_return_dispersion_1d", 1, 0, "all_9_sectors_S2"),
    ("nine_sector_rank_or_dispersion_20d", 20, 0, "all_9_sectors_S2"),
    ("correlation_involving_XLF_60d", 60, 0, "dependent_targets_S0_or_S2"),
]
impact_rows, impact_date_rows = [], []
event_position = spy_calendar.get_loc(XLF_EVENT)
for name, lookback, skip, affected_targets in impact_specs:
    dates = spy_calendar[event_position + skip:event_position + lookback]
    impact_rows.append(dict(
        dependency=name, affected_targets=affected_targets,
        lookback=lookback, skip_recent=skip, affected_rows=len(dates),
        first_affected=dates.min(), last_affected=dates.max(),
        invalid_reason="window_contains_xlf_unverified_event_return"))
    impact_date_rows.extend(dict(
        date=d, dependency=name, affected_targets=affected_targets,
        invalid_reason="window_contains_xlf_unverified_event_return") for d in dates)
xlf_dependency_impact = pd.DataFrame(impact_rows)
xlf_impact_dates = pd.DataFrame(impact_date_rows)

k = spy_calendar.get_loc(XLF_EVENT)
xlf_signal_timing = pd.DataFrame([dict(
    ticker="XLF", signal_date=spy_calendar[k-2],
    entry_date=spy_calendar[k-1], exit_date=XLF_EVENT, label_valid=False,
    invalid_reason="holding_interval_crosses_xlf_in_kind_distribution")])
assert xlf_signal_timing.loc[0, "signal_date"] == pd.Timestamp("2016-09-15")
display(xlf_dependency_impact)


## VIX and risk-free rates

Use the VIX close on SPY dates without filling missing observations.

In [ ]:
vix_raw = pd.read_csv(RAW / f"VIX_{SUFFIX}_cboe.csv")
vix_native = normalise_dates(vix_raw, "DATE", "VIX").rename(columns={
    "OPEN": "open", "HIGH": "high", "LOW": "low", "CLOSE": "close"})
vix_native = numeric_columns(vix_native, ["open", "high", "low", "close"], "VIX")
vix_native = isolate_bad_ohlc(vix_native, "VIX")
vix_extra = vix_native.loc[~vix_native["date"].isin(spy_calendar)]
for row in vix_extra.itertuples():
    calendar_changes.append(dict(asset="VIX", date=row.date, action="exclude_non_spy_date",
                                 reason="VIX_feature_calendar_is_SPY"))
vix = vix_native.set_index("date").reindex(spy_calendar)[["close"]].rename(
    columns={"close": "vix_close"}).reset_index()
vix["vix_close_valid"] = vix["vix_close"].notna()
vix["available_at"] = (
    (vix["date"] + pd.Timedelta(hours=18)).dt.tz_localize(NY).astype(str))
vix.loc[~vix["vix_close_valid"], "available_at"] = None
assert len(vix_extra) == 33 and vix["vix_close_valid"].all()
assert pd.isna(vix_native.set_index("date").at[pd.Timestamp("2006-02-08"), "open"])
log_step("calendar", "VIX", "date", "exclude_non_spy_dates", len(vix_extra),
         "Retain VIX close on the SPY calendar without forward filling.")


### Treasury-rate observations

In [ ]:
fed_exception_specs = [
    ("2008-12-26 2009-01-20 2009-12-24 2010-02-08 2010-02-09 "
     "2012-10-29 2012-10-30 2013-03-06 2013-12-10 2014-01-21 "
     "2014-02-13 2014-03-03 2014-03-17 2014-12-26 2015-02-17 "
     "2015-03-05 2016-01-22 2016-01-25 2016-01-26 2019-01-14 2019-02-20",
     "H.15 weather, inauguration or special-holiday closure",
     "https://www.federalreserve.gov/feeds/h15.html"),
    ("2007-01-02", "Board closure for the Ford national day of mourning",
     "https://www.federalreserve.gov/newsevents/pressreleases/other20061228a.htm"),
    ("2007-12-24", "Special Board Christmas closure",
     "https://www.federalreserve.gov/newsevents/pressreleases/other20071207a.htm"),
    ("2017-01-20", "Inauguration-day publication suspension",
     "https://www.federalreserve.gov/newsevents/2017-january.htm"),
    ("2018-12-05 2018-12-24", "National mourning or special Christmas publication suspension",
     "https://www.federalreserve.gov/newsevents/2018-december.htm"),
    ("2019-12-24", "Special Christmas publication suspension",
     "https://www.federalreserve.gov/newsevents/2019-December.htm"),
    ("2020-12-24", "Special Christmas publication suspension",
     "https://www.federalreserve.gov/newsevents/2020-december.htm"),
    ("2021-01-20", "Inauguration-day publication suspension",
     "https://www.federalreserve.gov/newsevents/2021-january.htm"),
    ("2024-12-24", "Special Christmas publication suspension",
     "https://www.federalreserve.gov/newsevents/2024-december.htm"),
    ("2025-01-09", "Publication suspension for the Carter national day of mourning",
     "https://www.federalreserve.gov/newsevents/2025-january.htm"),
    ("2025-12-24 2025-12-26", "Special Christmas publication suspension",
     "https://www.federalreserve.gov/newsevents/2025-december.htm"),
]
fed_exceptions = pd.DataFrame([
    dict(date=pd.Timestamp(d), reason=reason, source_url=url)
    for dates, reason, url in fed_exception_specs for d in dates.split()
]).sort_values("date").reset_index(drop=True)
assert len(fed_exceptions) == 33 and fed_exceptions["date"].is_unique
all_days = pd.date_range(START_DATE - pd.Timedelta(days=15),
                         END_DATE + pd.Timedelta(days=20), freq="D")
federal_holidays = USFederalHolidayCalendar().holidays(
    start=all_days.min(), end=all_days.max(), return_name=True)
publish_calendar = pd.DataFrame({"date": all_days})
publish_calendar["federal_holiday"] = publish_calendar["date"].map(federal_holidays).fillna("")
publish_calendar["extra_closure"] = publish_calendar["date"].map(
    fed_exceptions.set_index("date")["reason"]).fillna("")
publish_calendar["is_publication_day"] = (
    (publish_calendar["date"].dt.dayofweek < 5)
    & publish_calendar["federal_holiday"].eq("")
    & publish_calendar["extra_closure"].eq(""))
publish_dates = pd.DatetimeIndex(publish_calendar.loc[
    publish_calendar["is_publication_day"], "date"])
assert pd.Timestamp("2021-06-18") not in publish_dates
assert pd.Timestamp("2026-07-03") not in publish_dates
assert pd.Timestamp("2018-03-21") in publish_dates

fred_raw = pd.read_csv(RAW / f"DGS3MO_{SUFFIX}_fred.csv", dtype="str", keep_default_na=False)
fred = normalise_dates(fred_raw, "observation_date", "DGS3MO")
fred["raw_value"] = fred["DGS3MO"]
tokens = fred["DGS3MO"].str.strip()
missing_tokens = tokens.isin(["", "."])
fred["annual_yield_pct"] = pd.to_numeric(tokens.mask(missing_tokens), errors="raise")
if np.isinf(fred["annual_yield_pct"].to_numpy()).any():
    raise ValueError("DGS3MO contains infinite values.")
for row in fred.loc[missing_tokens].itertuples():
    changes.append(dict(asset="DGS3MO", date=row.date, field="DGS3MO",
                        original_value=repr(row.DGS3MO), cleaned_value=None,
                        rule="missing_marker_to_null"))
log_step("text", "DGS3MO", "DGS3MO", "strip_whitespace", tokens.ne(fred["DGS3MO"]).sum(),
         "Strip numeric whitespace; preserve the original string in raw_value.")
log_step("missing", "DGS3MO", "annual_yield_pct", "marker_to_null",
         missing_tokens.sum(), "Convert recognized missing markers to null and retain valid zeros.")
positions = publish_dates.searchsorted(pd.DatetimeIndex(fred["date"]), side="right")
assert positions.max() < len(publish_dates)
fred["assumed_release_date"] = publish_dates.take(positions).to_numpy()
fred["observation_date"] = fred["date"]
fred["observation_on_spy_date"] = fred["date"].isin(spy_calendar)
fred["rate_observed"] = fred["annual_yield_pct"].notna()
fred["invalid_reason"] = np.where(fred["rate_observed"], "", "source_missing_marker")
fred_observations = fred[[
    "observation_date", "raw_value", "annual_yield_pct", "assumed_release_date",
    "observation_on_spy_date", "rate_observed", "invalid_reason"]].copy()
assert missing_tokens.sum() == 222
assert (fred["observation_date"] < fred["assumed_release_date"]).all()


### Rate matching

Match rates released before entry and accrue cash returns on an ACT/365 basis.

In [ ]:
rate_updates = (fred_observations.loc[fred_observations["rate_observed"]]
                .sort_values(["assumed_release_date", "observation_date"]))
rate_updates = rate_updates.drop_duplicates("assumed_release_date", keep="last")
intervals = pd.DataFrame({"entry_date": spy_calendar})
intervals["exit_date"] = intervals["entry_date"].shift(-1)
risk_free = pd.merge_asof(
    intervals.sort_values("entry_date"),
    rate_updates[["observation_date", "annual_yield_pct", "assumed_release_date"]],
    left_on="entry_date", right_on="assumed_release_date",
    direction="backward", allow_exact_matches=False)
risk_free["holding_calendar_days"] = (
    risk_free["exit_date"] - risk_free["entry_date"]).dt.days.astype("Int64")
risk_free["rate_age_days"] = (
    risk_free["entry_date"] - risk_free["observation_date"]).dt.days.astype("Int64")
risk_free["rate_stale_over_7d"] = risk_free["rate_age_days"].gt(7).fillna(False)
risk_free["rate_known_at_entry"] = risk_free["annual_yield_pct"].notna()
risk_free["interval_valid"] = (
    risk_free["rate_known_at_entry"] & risk_free["exit_date"].notna())
risk_free["risk_free_return_act365"] = (
    risk_free["annual_yield_pct"] / 100
    * risk_free["holding_calendar_days"].astype("float64") / 365
).where(risk_free["interval_valid"])
risk_free["invalid_reason"] = ""
risk_free.loc[~risk_free["rate_known_at_entry"], "invalid_reason"] = "no_known_rate_yet"
risk_free.loc[risk_free["exit_date"].isna(), "invalid_reason"] = "next_open_outside_data_cutoff"
matched = risk_free["rate_known_at_entry"]
assert (risk_free.loc[matched, "assumed_release_date"]
        < risk_free.loc[matched, "entry_date"]).all()
assert (risk_free.loc[risk_free["interval_valid"], "holding_calendar_days"] > 0).all()

example = risk_free.set_index("entry_date").loc[pd.Timestamp("2025-10-14")]
assert example["observation_date"] == pd.Timestamp("2025-10-09")
assert np.isclose(example["annual_yield_pct"], 4.03)
rate_matching_checks = risk_free.loc[
    risk_free["entry_date"].between("2025-10-10", "2025-10-16")
    | risk_free["entry_date"].between("2016-01-21", "2016-01-29")].copy()


## Validation and export

Compare cash-close returns with the vendor-adjusted reference, accounting for different dividend conventions.

In [ ]:
CC_REVIEW_THRESHOLD_BPS = 0.1
cc_parts, return_coverage_rows = [], []
for ticker in ETFS:
    prices = etf_aligned[ticker]
    calculated = returns_by_ticker[ticker]
    reference_name = f"{ticker}_{SUFFIX}_yfinance_adjusted.csv"
    reference = pd.read_csv(RAW / reference_name)
    reference_dates = (
        pd.to_datetime(reference["Date"], utc=True, errors="raise")
        .dt.tz_convert(NY).dt.tz_localize(None).dt.normalize())
    if reference_dates.duplicated().any():
        raise ValueError(f"{ticker}: duplicate dates in the adjusted-price reference.")
    reference_close = pd.Series(
        pd.to_numeric(reference["Close"], errors="raise").to_numpy(),
        index=pd.DatetimeIndex(reference_dates)).reindex(spy_calendar)
    adjusted_return = reference_close / reference_close.shift(1) - 1
    comparable = (
        calculated["ret_cc_cash_valid"] & adjusted_return.notna()
        & ~prices["special_action"])
    return_coverage_rows.append(dict(
        check_type="close_cash_vs_adjusted", ticker=ticker,
        calendar_rows=len(spy_calendar),
        target_valid_rows=int(calculated["ret_cc_cash_valid"].sum()),
        compared_rows=int(comparable.sum()),
        valid_target_without_reference=int(
            (calculated["ret_cc_cash_valid"] & ~comparable).sum()),
        invalid_target_rows=int((~calculated["ret_cc_cash_valid"]).sum()),
        excluded_special_action_rows=int(prices["special_action"].sum())))
    comparison = pd.DataFrame(index=spy_calendar)
    comparison["ticker"] = ticker
    comparison["source_reference_csv"] = reference_name
    comparison["previous_date"] = calculated["previous_date"]
    comparison["previous_close"] = prices["close"].shift(1)
    comparison["dividend"] = prices["dividends"]
    comparison["cash_close_return"] = calculated["ret_cc_cash"]
    comparison["adjusted_close_return"] = adjusted_return
    comparison = comparison.loc[comparable].copy()
    comparison["dividend_day"] = comparison["dividend"].ne(0)
    fraction = comparison["dividend"] / comparison["previous_close"]
    if not fraction.between(0, 1, inclusive="left").all():
        raise ValueError(f"{ticker}: dividend-to-previous-close ratio requires review.")
    comparison["cash_minus_adjusted_bps"] = (
        comparison["cash_close_return"] - comparison["adjusted_close_return"]) * 10000
    comparison["theoretical_difference_bps"] = (
        -fraction * comparison["cash_close_return"] / (1 - fraction)) * 10000
    comparison["residual_bps"] = (
        comparison["cash_minus_adjusted_bps"] - comparison["theoretical_difference_bps"])
    comparison["review_over_threshold"] = (
        comparison["residual_bps"].abs() > CC_REVIEW_THRESHOLD_BPS)
    cc_parts.append(comparison.reset_index())

close_return_crosscheck = pd.concat(cc_parts, ignore_index=True)
cc_summary_rows = []
for ticker in ["ALL_ETFS"] + ETFS:
    group = (close_return_crosscheck if ticker == "ALL_ETFS" else
             close_return_crosscheck.loc[close_return_crosscheck["ticker"].eq(ticker)])
    for label, subset in [
        ("all", group), ("dividend_day", group.loc[group["dividend_day"]]),
        ("non_dividend_day", group.loc[~group["dividend_day"]])]:
        if subset.empty:
            continue
        raw_abs, residual_abs = (
            subset["cash_minus_adjusted_bps"].abs(), subset["residual_bps"].abs())
        cc_summary_rows.append(dict(
            ticker=ticker, day_group=label, compared_rows=len(subset),
            median_abs_difference_bps=raw_abs.median(),
            p99_abs_difference_bps=raw_abs.quantile(0.99),
            max_abs_difference_bps=raw_abs.max(),
            max_abs_residual_bps=residual_abs.max(),
            review_rows=int(subset["review_over_threshold"].sum())))
close_return_crosscheck_summary = pd.DataFrame(cc_summary_rows)
extremes = []
for metric in ["cash_minus_adjusted_bps", "residual_bps"]:
    selected = close_return_crosscheck.loc[
        close_return_crosscheck[metric].abs().nlargest(10).index].copy()
    selected["ranking_metric"] = metric
    extremes.append(selected)
close_return_crosscheck_extremes = pd.concat(extremes, ignore_index=True)
display(close_return_crosscheck_summary.query("ticker == 'ALL_ETFS'"))


### Open-return identity

Verify $1+r^{OO}_t=(1+r^{ID}_{t-1})(1+r^{ON}_t)$.

In [ ]:
OO_IDENTITY_ATOL = 1e-12
identity_parts = []
for ticker in ETFS:
    values = returns_by_ticker[ticker]
    previous_intraday = values["ret_intraday_price"].shift(1)
    comparable = (values["ret_oo_cash_valid"] & previous_intraday.notna()
                  & values["ret_overnight_cash_valid"])
    return_coverage_rows.append(dict(
        check_type="open_return_identity", ticker=ticker,
        calendar_rows=len(spy_calendar),
        target_valid_rows=int(values["ret_oo_cash_valid"].sum()),
        compared_rows=int(comparable.sum()),
        valid_target_without_reference=int(
            (values["ret_oo_cash_valid"] & ~comparable).sum()),
        invalid_target_rows=int((~values["ret_oo_cash_valid"]).sum()),
        excluded_special_action_rows=int(etf_aligned[ticker]["special_action"].sum())))
    identity = pd.DataFrame(index=spy_calendar)
    identity["ticker"] = ticker
    identity["previous_date"] = values["previous_date"]
    identity["cash_open_return"] = values["ret_oo_cash"]
    identity["previous_intraday_return"] = previous_intraday
    identity["current_overnight_cash_return"] = values["ret_overnight_cash"]
    identity["reconstructed_open_return"] = (
        (1 + previous_intraday) * (1 + values["ret_overnight_cash"]) - 1)
    identity = identity.loc[comparable].copy()
    identity["residual"] = (
        identity["cash_open_return"] - identity["reconstructed_open_return"])
    identity["within_tolerance"] = identity["residual"].abs() <= OO_IDENTITY_ATOL
    identity_parts.append(identity.reset_index())
open_return_identity = pd.concat(identity_parts, ignore_index=True)
open_return_identity_summary = open_return_identity.groupby("ticker", sort=False).agg(
    compared_rows=("date", "size"),
    max_abs_residual=("residual", lambda x: x.abs().max()),
    failed_rows=("within_tolerance", lambda x: int((~x).sum()))).reset_index()
open_return_identity_summary = pd.concat([
    pd.DataFrame([dict(
        ticker="ALL_ETFS", compared_rows=len(open_return_identity),
        max_abs_residual=open_return_identity["residual"].abs().max(),
        failed_rows=int((~open_return_identity["within_tolerance"]).sum()))]),
    open_return_identity_summary], ignore_index=True)
open_return_identity_failures = open_return_identity.loc[
    ~open_return_identity["within_tolerance"]].copy()
return_check_coverage = pd.DataFrame(return_coverage_rows)


### Opening-price diagnostics

In [ ]:
OPEN_NEAR_ZERO_THRESHOLD = 1e-6
gap_parts = []
for ticker in SECTORS:
    prices = etf_aligned[ticker]
    previous_close, previous_open = prices["close"].shift(1), prices["open"].shift(1)
    gaps = pd.DataFrame(index=spy_calendar)
    gaps["ticker"] = ticker
    gaps["year"] = gaps.index.year
    gaps["gap_valid"] = prices["open"].notna() & previous_close.notna()
    raw_gap = prices["open"] / previous_close - 1
    gaps["exact_zero"] = gaps["gap_valid"] & raw_gap.eq(0)
    gaps["near_zero"] = gaps["gap_valid"] & raw_gap.abs().le(OPEN_NEAR_ZERO_THRESHOLD)
    gaps["open_pair_valid"] = prices["open"].notna() & previous_open.notna()
    gaps["repeated_open"] = gaps["open_pair_valid"] & prices["open"].eq(previous_open)
    gap_parts.append(gaps.reset_index())
gap_diagnostics = pd.concat(gap_parts, ignore_index=True)

def summarise_gaps(frame):
    return dict(
        rows=len(frame), valid_gap_rows=int(frame["gap_valid"].sum()),
        exact_zero_gap_rows=int(frame["exact_zero"].sum()),
        near_zero_gap_rows=int(frame["near_zero"].sum()),
        comparable_open_pairs=int(frame["open_pair_valid"].sum()),
        repeated_open_rows=int(frame["repeated_open"].sum()))

annual_rows = []
for (ticker, year), group in gap_diagnostics.groupby(["ticker", "year"], sort=True):
    annual_rows.append(dict(ticker=ticker, year=int(year), **summarise_gaps(group)))
opening_gap_year_review = pd.DataFrame(annual_rows)
period_specs = [
    ("2006-2010", "descriptive", "2006-01-01", "2010-12-31"),
    ("2011-2026Sep", "descriptive", "2011-01-01", "2026-09-30"),
    ("train_2008-2010", "training_early", "2008-01-01", "2010-12-31"),
    ("train_2011-2017", "training_later", "2011-01-01", "2017-12-31"),
    ("validation_2018-2022", "validation", "2018-01-01", "2022-12-31"),
    ("test_2023-2026Sep", "test_descriptive_only", "2023-01-01", "2026-09-30"),
]
period_rows = []
for ticker in ["ALL_9_SECTORS"] + SECTORS:
    asset_rows = (gap_diagnostics if ticker == "ALL_9_SECTORS" else
                  gap_diagnostics.loc[gap_diagnostics["ticker"].eq(ticker)])
    for label, role, start, end in period_specs:
        subset = asset_rows.loc[asset_rows["date"].between(start, end)]
        period_rows.append(dict(
            ticker=ticker, period=label, role=role, **summarise_gaps(subset)))
opening_gap_period_review = pd.DataFrame(period_rows)
for table in [opening_gap_year_review, opening_gap_period_review]:
    table["near_zero_gap_fraction"] = table["near_zero_gap_rows"] / table["valid_gap_rows"]
    table["repeated_open_fraction"] = table["repeated_open_rows"] / table["comparable_open_pairs"]

reference_path = ROOT / "notebooks" / "quality_checks" / "opening_gap_by_year.csv"
opening_reference_hash = sha256(reference_path)
reference_counts = pd.read_csv(reference_path).query(
    "version == 'raw_returned' and asset in @SECTORS").rename(columns={"asset": "ticker"})
count_columns = ["rows", "valid_gap_rows", "exact_zero_gap_rows",
                 "near_zero_gap_rows", "comparable_open_pairs", "repeated_open_rows"]
opening_gap_01_crosscheck = opening_gap_year_review[["ticker", "year"] + count_columns].merge(
    reference_counts[["ticker", "year"] + count_columns],
    on=["ticker", "year"], how="outer", suffixes=("_02", "_01"),
    indicator=True, validate="one_to_one")
count_match = opening_gap_01_crosscheck["_merge"].eq("both")
for col in count_columns:
    count_match &= opening_gap_01_crosscheck[f"{col}_02"].eq(
        opening_gap_01_crosscheck[f"{col}_01"])
opening_gap_01_crosscheck["counts_match"] = count_match
display(opening_gap_period_review.query("ticker == 'ALL_9_SECTORS'")[
    ["period", "valid_gap_rows", "near_zero_gap_rows", "near_zero_gap_fraction",
     "repeated_open_fraction"]])


### Data integrity

In [ ]:
checks = []
def check(name, passed, detail):
    checks.append(dict(check=name, passed=bool(passed), detail=str(detail)))
    if not passed:
        raise AssertionError(f"{name}: {detail}")

check("all_etf_keys_unique", not etf_prices.duplicated(["date", "ticker"]).any(), len(etf_prices))
check("all_etf_calendars_match_spy",
      all(d.index.equals(spy_calendar) for d in etf_aligned.values()), "15 × 5218")
for ticker in ETFS:
    a = etf_aligned[ticker]
    n = etf_native[ticker].set_index("date")
    for field in ["open", "high", "low", "close", "volume", "dividends", "stock_splits"]:
        pd.testing.assert_series_equal(a.loc[n.index, field], n[field], check_names=False)
check("existing_etf_values_preserved", True, "Observed values are unchanged by calendar alignment.")
hyg = etf_aligned["HYG"]
pre_hyg = hyg.index < pd.Timestamp("2007-04-11")
check("hyg_precoverage_318_missing", int(pre_hyg.sum()) == 318
      and hyg.loc[pre_hyg, ["open", "close", "volume", "dividends"]].isna().all().all(), 318)
check("hyg_first_overnight_return_missing",
      pd.isna(returns_by_ticker["HYG"].at[pd.Timestamp("2007-04-11"), "ret_oo_cash"]),
      "No previous SPY open exists on the first observation date.")
check("vix_close_preserved",
      np.array_equal(vix["vix_close"].to_numpy(),
                     vix_native.set_index("date").loc[spy_calendar, "close"].to_numpy()),
      "Exclude extra dates without changing close values.")
check("all_rate_zeros_preserved", fred_observations["annual_yield_pct"].eq(0).sum() == 21, 21)
check("no_future_rate_match",
      (risk_free.loc[matched, "assumed_release_date"]
       < risk_free.loc[matched, "entry_date"]).all(), int(matched.sum()))
check("valid_non_spy_rate_observations_preserved",
      ((~fred_observations["observation_on_spy_date"])
       & fred_observations["rate_observed"]).sum() == 10, 10)

xlf_r = returns_by_ticker["XLF"]
check("xlf_only_cross_event_returns_invalid",
      xlf_r.loc[XLF_EVENT, cross_names].isna().all()
      and pd.notna(xlf_r.at[XLF_EVENT, "ret_intraday_price"])
      and pd.notna(xlf_r.at[pd.Timestamp("2016-09-20"), "ret_oo_cash"]),
      "Exclude three cross-event returns; retain intraday and subsequent returns.")
check("spy_not_invalidated_by_xlf",
      returns_by_ticker["SPY"].loc[XLF_EVENT, cross_names].notna().all(), "SPY remains valid.")
mom = xlf_dependency_impact.set_index("dependency").loc["momentum_12_1"]
check("momentum_exact_impact_interval",
      mom["affected_rows"] == 231 and mom["first_affected"] == pd.Timestamp("2016-10-18")
      and mom["last_affected"] == pd.Timestamp("2017-09-18"),
      "231 affected trading days; recover after the event leaves the window.")
check("raw_hashes_unchanged", {n: sha256(RAW/n) for n in expected_hashes} == expected_hashes, 47)
check("download_manifest_unchanged", sha256(DOWNLOAD_MANIFEST_PATH) == download_manifest_hash, True)
check("close_adjusted_reference_coverage",
      return_check_coverage.query("check_type == 'close_cash_vs_adjusted'")[
          "valid_target_without_reference"].eq(0).all(),
      f"{len(close_return_crosscheck)} cash-close intervals have reference coverage.")
check("open_cash_return_identity",
      len(open_return_identity) > 0 and open_return_identity["within_tolerance"].all(),
      f"{len(open_return_identity)} intervals; maximum absolute residual "
      f"{open_return_identity['residual'].abs().max():.4g}; tolerance {OO_IDENTITY_ATOL}")
check("opening_gap_year_counts_match_01",
      len(opening_gap_01_crosscheck) == len(opening_gap_year_review)
      and opening_gap_01_crosscheck["counts_match"].all(),
      f"{len(opening_gap_year_review)} asset-years; six annual counts agree.")
check("opening_gap_reference_unchanged",
      sha256(reference_path) == opening_reference_hash, "Download-stage annual diagnostics are unchanged.")
validation_checks = pd.DataFrame(checks)


### Export

In [ ]:
file_records = []

def save_table(frame, path, kind):
    out = frame.copy()
    for col in out.columns:
        if is_datetime64_any_dtype(out[col]):
            out[col] = out[col].dt.strftime("%Y-%m-%d")
    out.to_csv(path, index=False, encoding="utf-8", float_format="%.17g")
    if kind == "interim":
        restored = pd.read_csv(path, low_memory=False, float_precision="round_trip")
        for field in frame.select_dtypes(include=["number"]).columns:
            expected = frame[field].to_numpy(dtype="float64", na_value=np.nan)
            actual = pd.to_numeric(restored[field], errors="raise").to_numpy(
                dtype="float64", na_value=np.nan)
            np.testing.assert_array_equal(actual, expected)
    dates = [c for c in ["date", "signal_date", "entry_date", "observation_date"] if c in frame]
    date_col = dates[0] if dates else None
    file_records.append(dict(
        file=path.relative_to(ROOT).as_posix(), kind=kind, rows=len(frame),
        columns=len(frame.columns),
        first_date=(frame[date_col].min().strftime("%Y-%m-%d") if date_col and len(frame)
                    and pd.notna(frame[date_col].min()) else ""),
        last_date=(frame[date_col].max().strftime("%Y-%m-%d") if date_col and len(frame)
                   and pd.notna(frame[date_col].max()) else ""),
        sha256=sha256(path)))

data_tables = {
    "etf_prices_aligned.csv": etf_prices,
    "etf_base_returns.csv": base_returns,
    "vix_aligned.csv": vix,
    "dgs3mo_observations.csv": fred_observations,
    "risk_free_open_intervals.csv": risk_free,
}
for name, table in data_tables.items():
    save_table(table, INTERIM / name, "interim")

summary_rows = []
for ticker in ETFS:
    d = etf_aligned[ticker]
    summary_rows.append(dict(
        asset=ticker, native_rows=native_rows[ticker], aligned_rows=len(d),
        first_observation=etf_native[ticker]["date"].min(),
        last_observation=etf_native[ticker]["date"].max(),
        added_empty_rows=int((~d["observed"]).sum()), excluded_dates=0,
        usable_close_rows=int(d["close"].notna().sum())))
summary_rows += [
    dict(asset="VIX", native_rows=len(vix_raw), aligned_rows=len(vix),
         first_observation=vix_native["date"].min(), last_observation=vix_native["date"].max(),
         added_empty_rows=0, excluded_dates=len(vix_extra), usable_close_rows=len(vix)),
    dict(asset="DGS3MO", native_rows=len(fred_raw), aligned_rows=len(risk_free),
         first_observation=fred["date"].min(), last_observation=fred["date"].max(),
         added_empty_rows=0, excluded_dates=0, usable_close_rows=np.nan),
]
cleaning_summary = pd.DataFrame(summary_rows)
missing_rows = []
for ticker in ETFS:
    d = etf_aligned[ticker]
    for field in list(field_map.values()):
        missing_rows.append(dict(
            asset=ticker, field=field, null_rows=int(d[field].isna().sum()),
            null_on_observed_row=int((d[field].isna() & d["observed"]).sum())))
missing_after_alignment = pd.DataFrame(missing_rows)
detail_columns = ["asset", "date", "field", "original_value", "cleaned_value", "rule"]
change_table = pd.DataFrame(changes, columns=detail_columns)
calendar_change_table = pd.DataFrame(calendar_changes)
report_tables = {
    "cleaning_summary.csv": cleaning_summary,
    "cleaning_log.csv": pd.DataFrame(steps),
    "value_changes.csv": change_table,
    "calendar_changes.csv": calendar_change_table,
    "vix_native_cleaned_for_audit.csv": vix_native,
    "missing_after_alignment.csv": missing_after_alignment,
    "corporate_action_checks.csv": corporate_action_checks,
    "dividend_unit_checks.csv": dividend_unit_checks,
    "dividend_open_return_comparison.csv": dividend_return_comparison,
    "xlf_dependency_impact.csv": xlf_dependency_impact,
    "xlf_impact_dates.csv": xlf_impact_dates,
    "xlf_signal_timing_check.csv": xlf_signal_timing,
    "risk_free_calendar_exceptions.csv": fed_exceptions,
    "risk_free_publication_calendar.csv": publish_calendar,
    "risk_free_matching_checks.csv": rate_matching_checks,
    "validation_checks.csv": validation_checks,
    "close_return_crosscheck.csv": close_return_crosscheck,
    "close_return_crosscheck_summary.csv": close_return_crosscheck_summary,
    "close_return_crosscheck_extremes.csv": close_return_crosscheck_extremes,
    "open_return_identity_summary.csv": open_return_identity_summary,
    "open_return_identity_failures.csv": open_return_identity_failures,
    "return_check_coverage.csv": return_check_coverage,
    "opening_gap_by_year_review.csv": opening_gap_year_review,
    "opening_gap_by_period_review.csv": opening_gap_period_review,
    "opening_gap_01_crosscheck.csv": opening_gap_01_crosscheck,
}
for name, table in report_tables.items():
    save_table(table, CLEANING_REPORTS / name, "quality_report")

descriptions = {
    "date": "New York trading date on the SPY master calendar.",
    "ticker": "Asset ticker; sector, market and auxiliary ETFs have distinct roles.",
    "open": "Open in supplier split-normalized units; not dividend-adjusted.",
    "high": "High in supplier split-normalized units; not dividend-adjusted.",
    "low": "Low in supplier split-normalized units; not dividend-adjusted.",
    "close": "Close in supplier split-normalized units; not dividend-adjusted.",
    "adj_close": "Supplier-adjusted close from the same raw snapshot.",
    "volume": "Volume in supplier share units; not rescaled for dividends.",
    "dividends": "Cash dividend per share on the ex-date in supplier units; zero means no event.",
    "stock_splits": "Reported action ratio; XLF 1.231 is not an ordinary split.",
    "capital_gains": "Capital-gain distribution; currently zero; nonzero observations require review.",
    "adjustment_factor": "adj_close / close from the same snapshot; for adjustment checks.",
    "special_action": "XLF in-kind distribution on 2016-09-19.",
    "volume_zero": "Observed zero volume, distinct from missing volume.",
    "observed": "True for an original observation; false for an added calendar row.",
    "ohlc_valid": "All OHLC fields are present; cross-event returns may still be invalid.",
    "quote_available_at": "Assumed quote availability at 18:00 New York time, with timezone.",
    "available_at": "Assumed VIX-close availability at 18:00 New York time, with timezone.",
    "invalid_reason": "Quality or invalidity reason; metadata, not a predictor.",
    "previous_date": "Previous SPY trading date, not the previous nonmissing observation.",
    "ret_cc_cash": "(current close + dividend) / previous SPY-date close - 1; exit-date indexed.",
    "ret_oo_cash": "(current open + dividend) / previous SPY-date open - 1; exit-date indexed.",
    "ret_overnight_cash": "(current open + dividend) / previous SPY-date close - 1; cash accrual.",
    "ret_intraday_price": "current close / current open - 1; no entry-day ex-dividend entitlement.",
    "vix_close": "Official Cboe VIX close level, not an investable asset return.",
    "vix_close_valid": "Whether VIX close is available.",
    "observation_date": "FRED rate observation date, distinct from its availability date.",
    "raw_value": "Original FRED string retained for audit, not a predictor.",
    "annual_yield_pct": "DGS3MO annual percentage yield; 4.03 means 4.03%; zero is valid.",
    "assumed_release_date": "First H.15 publication day after observation; available after that day ends.",
    "observation_on_spy_date": "Whether the native observation date is a SPY trading date.",
    "rate_observed": "Whether the source rate is observed; zero is valid.",
    "entry_date": "Entry-open date used to match a previously released rate.",
    "exit_date": "Next SPY open; missing at the data cutoff without October extension.",
    "holding_calendar_days": "Calendar days in the open-to-open interval; Friday to Monday is three.",
    "rate_age_days": "Calendar days from rate observation to entry.",
    "rate_stale_over_7d": "Flag for a rate older than seven days; no automatic exclusion.",
    "rate_known_at_entry": "Whether a released rate is available under the stated timing assumption.",
    "interval_valid": "Both a known rate and an interval exit date are available.",
    "risk_free_return_act365": "annual_yield_pct / 100 * calendar days / 365; cash-return proxy.",
}
dictionary_rows = []
for name, table in data_tables.items():
    for col in table:
        if col in descriptions:
            explanation = descriptions[col]
        elif col.startswith("adjusted_"):
            explanation = "Raw field * adjustment_factor from the same snapshot; verification only."
        elif col.endswith("_invalid_reason"):
            explanation = "Invalidity reason for the corresponding return; not a predictor."
        elif col.endswith("_valid"):
            explanation = "Validity of the corresponding return; complete windows require every needed day."
        else:
            raise KeyError(f"Missing field definition: {name} / {col}")
        dictionary_rows.append(dict(
            file=f"data/interim/{name}", column=col, dtype=str(table[col].dtype),
            null_rows=int(table[col].isna().sum()), definition=explanation))
data_dictionary = pd.DataFrame(dictionary_rows)
save_table(data_dictionary, CLEANING_REPORTS / "data_dictionary.csv", "documentation")
output_summary = pd.DataFrame(file_records)
assert output_summary["file"].is_unique, "Duplicate output paths."
output_summary.to_csv(CLEANING_REPORTS / "output_summary.csv", index=False, encoding="utf-8")

cleaning_manifest = dict(
    rule_version=RULE_VERSION, run_utc=datetime.now(timezone.utc).isoformat(),
    python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
    requested_start=str(START_DATE.date()), requested_end=str(END_DATE.date()),
    input_hashes=expected_hashes, input_download_manifest_sha256=download_manifest_hash,
    output_files=file_records, outputs_summary_sha256=sha256(CLEANING_REPORTS / "output_summary.csv"),
    diagnostic_thresholds=dict(
        cc_corrected_residual_review_bps=CC_REVIEW_THRESHOLD_BPS,
        oo_identity_absolute_tolerance=OO_IDENTITY_ATOL,
        opening_gap_near_zero_return=OPEN_NEAR_ZERO_THRESHOLD),
    diagnostic_input_hashes={
        "notebooks/quality_checks/opening_gap_by_year.csv": opening_reference_hash},
    cc_reference="separately downloaded yfinance_adjusted Close; same Yahoo supplier",
    cc_review_rows=int(close_return_crosscheck["review_over_threshold"].sum()),
    opening_decision="retain zero gaps and overnight/intraday returns; no automatic era-based removal or weighting",
    publication_calendar="USFederalHolidayCalendar + 33 verified extra nonpublication dates",
    rate_availability="first later publication date, usable only after that date ends",
    rate_limitations="assumed timing; final snapshot, not historical vintage; exceptions not guaranteed exhaustive",
    return_convention="cash distribution accrued at face value on ex-date; no automatic open reinvestment",
    special_action="XLF 2016-09-19 cross-event cash returns invalid; retain prices and calendar",
    next_stage="construct features and labels; apply dependency-specific masks when forming model panel")
(CLEANING_REPORTS / "cleaning_manifest.json").write_text(
    json.dumps(cleaning_manifest, ensure_ascii=False, indent=2), encoding="utf-8")


### Coverage summary

In [ ]:
print(f"Cleaned {len(ETFS)} ETFs on {len(spy_calendar):,} SPY trading dates; "
      f"{len(validation_checks)} checks passed.")
display(cleaning_summary)
display(output_summary.loc[output_summary["kind"].eq("interim"),
    ["file", "rows", "columns", "first_date", "last_date"]])
display(open_return_identity_summary.query("ticker == 'ALL_ETFS'"))
